# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [82]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [83]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


In [84]:
print("Original revenue:", (df['price'].str.replace('$', '').astype(float) *
                           pd.to_numeric(df['qty'], errors='coerce')).sum())

Original revenue: 4852.5


### TODO 1 — drop duplicates

In [85]:
# Assigning the dropped duplicates to a new, clean data set
df_clean = df.drop_duplicates()

# Determining the number of dropped columns - taking original length of dataset minus the length after duplicates were removed
dropped = len(df) - len(df_clean)
rows_affected = dropped

# Log for the given step
log(df_clean, "Dropped the duplicate rows", rows_affected)

[     order_id           item  category  qty   price
0           0    Rain Poncho  RainGear  3.0  $12.00
1           1    foam finger   Apparel  1.0     7.5
2           2  cheese burger     Merch  1.0   $7.50
3           3   Cheeseburger      Food  NaN   $7.50
4           4  cheese burger   Apparel  1.0     7.5
..        ...            ...       ...  ...     ...
295       295    rain poncho      food  3.0     6.0
296       296  cheese burger  RainGear -1.0     6.0
297       297    Foam Finger      food  1.0     7.5
298       298    Rain Poncho     Merch  1.0   $7.50
299       299  cheese burger      food  1.0  $12.00

[300 rows x 5 columns]] Dropped the duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [86]:
# Using a copy of the cleaned data
df_clean = df.drop_duplicates().copy()

# Taking the price and re-typing it to a float
df_clean['price'] = df_clean['price'].str.replace('$', '').astype(float)

# Determining the type of the price variable
price_type = df_clean['price'].dtype

# Log for the given step
log(df_clean, "Converted the price to a float type", price_type)

[     order_id           item  category  qty  price
0           0    Rain Poncho  RainGear  3.0   12.0
1           1    foam finger   Apparel  1.0    7.5
2           2  cheese burger     Merch  1.0    7.5
3           3   Cheeseburger      Food  NaN    7.5
4           4  cheese burger   Apparel  1.0    7.5
..        ...            ...       ...  ...    ...
295       295    rain poncho      food  3.0    6.0
296       296  cheese burger  RainGear -1.0    6.0
297       297    Foam Finger      food  1.0    7.5
298       298    Rain Poncho     Merch  1.0    7.5
299       299  cheese burger      food  1.0   12.0

[300 rows x 5 columns]] Converted the price to a float type (float64 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [87]:
# Taking the quantity and re-typing it to a numeric value
df_clean['qty'] = pd.to_numeric(df_clean['qty'], errors='coerce')

# Dropping rows with missing and negative quantities
df_clean = df_clean[df_clean['qty'].notna() & (df_clean['qty'] >= 1)]

# Determining the type of the quantity variable
qty_type = df_clean['qty'].dtype

# Log for the given step
log(df_clean, "Converted the qty to a numeric type & dropped rows with missing and negative values", qty_type)

[     order_id           item  category  qty  price
0           0    Rain Poncho  RainGear  3.0   12.0
1           1    foam finger   Apparel  1.0    7.5
2           2  cheese burger     Merch  1.0    7.5
4           4  cheese burger   Apparel  1.0    7.5
5           5    Foam Finger      food  3.0    6.0
..        ...            ...       ...  ...    ...
294       294    rain poncho     Merch  2.0    6.0
295       295    rain poncho      food  3.0    6.0
297       297    Foam Finger      food  1.0    7.5
298       298    Rain Poncho     Merch  1.0    7.5
299       299  cheese burger      food  1.0   12.0

[275 rows x 5 columns]] Converted the qty to a numeric type & dropped rows with missing and negative values (float64 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [88]:
# Printing the names for the item
print(df_clean['item'].value_counts())

# Create the mappings
ITEM_MAP = {
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho',
    'cheese burger': 'Cheeseburger',
    'Cheeseburger': 'Cheeseburger'
}

# Apply the mapping
df_clean['item'] = df_clean['item'].map(ITEM_MAP)

# Log the collapse
items_before = 6
items_after = df_clean['item'].nunique()

# Print the log
log(df_clean, "Item Names", items_before - items_after)

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[     order_id          item  category  qty  price
0           0   Rain Poncho  RainGear  3.0   12.0
1           1   Foam Finger   Apparel  1.0    7.5
2           2  Cheeseburger     Merch  1.0    7.5
4           4  Cheeseburger   Apparel  1.0    7.5
5           5   Foam Finger      food  3.0    6.0
..        ...           ...       ...  ...    ...
294       294   Rain Poncho     Merch  2.0    6.0
295       295   Rain Poncho      food  3.0    6.0
297       297   Foam Finger      food  1.0    7.5
298       298   Rain Poncho     Merch  1.0    7.5
299       299  Cheeseburger      food  1.0   12.0

[275 rows x 5 columns]] Item Names (3 row(s))


/tmp/ipykernel_2456/1884383633.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['item'] = df_clean['item'].map(ITEM_MAP)


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [89]:
# Printing the names for the category
print(df_clean['category'].value_counts())

# Create the mappings
CATEGORY_MAP = {
    'Food': 'Food',
    'food': 'Food',
    'rain-gear': 'RainGear',
    'RainGear': 'RainGear',
    'Apparel': 'Apparel',
    'Merch': 'Merch'
}

# Apply the mapping
df_clean['category'] = df_clean['category'].map(CATEGORY_MAP)

# Log the collapse
categories_before = 6
categories_after = df_clean['category'].nunique()

# Print the log
log(df_clean, "Normalized category names; kept Apparel and Merch separate", categories_before - categories_after)

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[     order_id          item  category  qty  price
0           0   Rain Poncho  RainGear  3.0   12.0
1           1   Foam Finger   Apparel  1.0    7.5
2           2  Cheeseburger     Merch  1.0    7.5
4           4  Cheeseburger   Apparel  1.0    7.5
5           5   Foam Finger      Food  3.0    6.0
..        ...           ...       ...  ...    ...
294       294   Rain Poncho     Merch  2.0    6.0
295       295   Rain Poncho      Food  3.0    6.0
297       297   Foam Finger      Food  1.0    7.5
298       298   Rain Poncho     Merch  1.0    7.5
299       299  Cheeseburger      Food  1.0   12.0

[275 rows x 5 columns]] Normalized category names; kept Apparel and Merch separate (2 row(s))


/tmp/ipykernel_2456/614964146.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['category'] = df_clean['category'].map(CATEGORY_MAP)


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [90]:
assert df_clean.duplicated().sum() == 0
assert df_clean['qty'].min() >= 1
assert df_clean['price'].dtype == float

assert df_clean['item'].nunique() == 3
assert df_clean['category'].nunique() == 4

print('clean:', df_clean.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [91]:
# Adding the revenue column - calculating revenue for each row
df_clean['revenue'] = df_clean['price'] * df_clean['qty']

# Calculating the revenue by category - sorting the highest to be first
revenue_by_category = df_clean.groupby('category')['revenue'].sum().sort_values(ascending=False)

# Printing revenue by category
print(revenue_by_category.round(2))

# Print overall revenue
print('Overall Revenue:', df_clean['revenue'].sum().round(2))

# Print the log
log(df_clean, "Calculated revenue using price * qty", df_clean['revenue'].sum().round(2))

category
Food        1656.0
RainGear    1512.0
Merch        856.5
Apparel      715.5
Name: revenue, dtype: float64
Overall Revenue: 4740.0
[     order_id          item  category  qty  price  revenue
0           0   Rain Poncho  RainGear  3.0   12.0     36.0
1           1   Foam Finger   Apparel  1.0    7.5      7.5
2           2  Cheeseburger     Merch  1.0    7.5      7.5
4           4  Cheeseburger   Apparel  1.0    7.5      7.5
5           5   Foam Finger      Food  3.0    6.0     18.0
..        ...           ...       ...  ...    ...      ...
294       294   Rain Poncho     Merch  2.0    6.0     12.0
295       295   Rain Poncho      Food  3.0    6.0     18.0
297       297   Foam Finger      Food  1.0    7.5      7.5
298       298   Rain Poncho     Merch  1.0    7.5      7.5
299       299  Cheeseburger      Food  1.0   12.0     12.0

[275 rows x 6 columns]] Calculated revenue using price * qty (4740.0 row(s))


/tmp/ipykernel_2456/402801694.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['revenue'] = df_clean['price'] * df_clean['qty']


**What I would tell the vendor:** I would tell the vendor to stock more food products, since food creates the most revenue. Ensuring all the food is stocked, it ensures that the vendor can maximize their revenue.

### TODO 8 — read back your log

In [92]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,order_id item category qty ...,Dropped the duplicate rows,15
1,order_id item category qty p...,Converted the price to a float type,float64
2,order_id item category qty pr...,Converted the qty to a numeric type & dropped ...,float64
3,order_id item category qty pr...,Item Names,3
4,order_id item category qty pr...,Normalized category names; kept Apparel and Me...,2
5,order_id item category qty pr...,Calculated revenue using price * qty,4740.0


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

**A:** The final reported revenue is 4,740.00 dollars, and before the changes were made, the original revenue was 4,852.50 dollars, thus a 112.50 dollar drop. Dropping the duplicates changed the total revenue the most because the rest of the changes did not impact the total revenue.

**B:** A reasonable person could have chosen to keep the duplicate rows, since technically it would have increased the reported revenue; however, I removed them because they are duplicates and thus overrepresenting the revenue. To ensure that records are correct and not counted more than once, I removed them.